In [1]:
# ========== Cell 1: 安装系统依赖 ==========
!sudo apt-get update
!sudo apt-get install -y zstd
!zstd --version
print("✅ zstd 安装成功！")

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [95.6 kB]
Get:10 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,129 kB]
Hit:13 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubun

In [2]:
!pip install torch_geometric


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.8 MB/s eta 0:00:00


In [3]:
# ========== Cell 2: 安装 Python 依赖 ==========
!pip install sentence-transformers
print("✅ sentence-transformers 安装完成！")


✅ sentence-transformers 安装完成！


In [4]:
# ========== Cell 3: 安装 Ollama ==========
!rm -rf /usr/local/bin/ollama
!rm -rf ~/.ollama
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version
print("✅ Ollama 安装成功！")


>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
]11;?\Warning: could not connect to a running Ollama instance
✅ Ollama 安装成功！


In [5]:
# ========== Cell 4: 启动 Ollama 并下载模型 ==========
import subprocess
import time

!pkill -9 -f "ollama"
time.sleep(3)

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(15)

print("正在下载 Qwen2-7B 模型...")
!ollama pull qwen2:7b
print("✅ 模型下载完成！")


正在下载 Qwen2-7B 模型...
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling 43f7a214e532:   0% ▕                  ▏ 1.8 MB/4.4 GB                  pulling manifest 
pulling 43f7a214e532:   1% ▕                  ▏  58 MB/4.4 GB                  pulling manifest 
pulling 43f7a214e532:   3% ▕                  ▏ 111 MB/4.4 GB                  pulling manifest 
pulling 43f7a214e532:   5% ▕                  ▏ 200 MB/4.4 GB                  pulling manifest 
pulling 43f7a214e532:   7% ▕█                 ▏ 293 MB/4.4 GB                  pulling manifest 
pulling 43f7a214e532:   8% ▕█                 ▏ 343 MB/4.4 GB                  pulling manifest 
pulling 43f7a214e532:  10% ▕█                 ▏ 442 MB/4.4 GB                  pulling manifest 
pulling 43f7a214e532:  12% ▕██                ▏ 546 MB/4.4 GB                  pulling manifest 
pulling 43f7a214e532:  13%

In [6]:
# ========== Cell 5: 测试 Ollama 连通性 ==========
import requests

response = requests.post(
    "http://localhost:11434/v1/chat/completions",
    json={
        "model": "qwen2:7b",
        "messages": [{"role": "user", "content": "Hello!"}],
        "max_tokens": 10
    }
)

print("🎉 LLM 响应正常！")
print("响应内容：", response.json()["choices"][0]["message"]["content"])


🎉 LLM 响应正常！
响应内容： Hello! How can I assist you today?


In [7]:
# ========== Cell 6: 克隆仓库并进入项目目录 ==========
!git clone https://github.com/pmateo-uc3m/GAMMAF.git
%cd /kaggle/working/GAMMAF

!mkdir -p data
!mkdir -p results


Cloning into 'GAMMAF'...
remote: Enumerating objects: 87, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 87 (delta 34), reused 54 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (87/87), 104.79 KiB | 2.33 MiB/s, done.
Resolving deltas: 100% (34/34), done.
/kaggle/working/GAMMAF


In [8]:
# ========== Cell 7: 在 GAMMAF 目录内创建 .env ==========
with open(".env", "w") as f:
    f.write("""BASE_URL="http://localhost:11434/v1"
MODEL_NAME="qwen2:7b"
API_KEY="ollama"
""")


In [9]:
# ========== Cell 8: 在 GAMMAF 目录内创建配置文件 ==========
import os

os.makedirs("config-examples", exist_ok=True)

with open("config-examples/kaggle-generation-config.yaml", "w") as f:
    f.write("""timeout: 60
parallel_questions: 4
prompts: prompts/prompts_blindguard.json
dataset_tag: GSM8K
questions_random_seed: 42
verbose: true

save_data_dir: /kaggle/working/GAMMAF/data
file_name: train-data-kaggle.pkl

process_text: false
clean_data: false
text_process_workers: 0

debate_config:
  num_agents: 6
  num_malicious: 2
  max_rounds: 3
  consensus_threshold: 1.0
  malicious_randomization_seed: 42

  n_questions: 50
  n_questions_random_topo: 30

  random_topo_seed: 24
  density:
    min: 0.3
    max: 0.7
""")

with open("config-examples/kaggle-evaluation-config.yaml", "w") as f:
    f.write("""models_directory: defense-models
output_file: /kaggle/working/GAMMAF/results/evaluation-kaggle.json

defense_model_train_configs:
  BlindGuard:
    pkl_train: /kaggle/working/GAMMAF/data/train-data-kaggle.pkl
    seed: 42
    device: cuda
    anomaly_rate: 0.2
    anomaly_scale: 0.5
    anomalize_data: true
    no_balance: false
    topologies: ["tree", "chain", "star", "random"]

    input_dim: 1152
    hidden_dim: 128
    emb_dim: 64
    batch_size: 128
    val_split: 0.2
    num_epochs: 10
    temperature: 0.07
    learning_rate: 0.001
    weight_decay: 0.0001
    scheduler_t_max: 10

  XG-Guard:
    pkl_train: /kaggle/working/GAMMAF/data/train-data-kaggle.pkl
    seed: 42
    device: cuda
    topologies: ["tree", "chain", "star", "random"] 
    feat_dim_s: 384
    feat_dim_t: 384
    hidden_dim: 128
    batch_size: 16
    val_split: 0.2
    num_epochs: 3
    learning_rate: 0.001
    weight_decay: 0.0001
    alpha: 0.5

live_evaluation_config:
  timeout: 60
  prompts_file: prompts/prompts_blindguard.json
  questions_path: DatasetManager.py
  questions_class_name: MMLULoader
  questions_dataset_tag: MMLU
  questions_random_seed: 28
  no_consensus_check: false 

  num_agents: 6
  num_malicious_agents: 2
  malicious_seed: 123
  max_rounds: 3
  consensus_threshold: 1.0
  check_consensus_only_unflagged: true

  top_k_defense: 1
  no_defense_baseline: false

  max_concurrent_inference: 4

  num_questions: 50
  new_random_each_question: true
  n_questions_on_random_topo: 30
  topologies_seed: 24
  density_range_for_random_topo: [0.3, 0.7]

  text_processor_path: TextProcessingManager.py
  text_processor_class_name: RoundProcessor

  save_traces: false
  clean_debates_with_empty_responses: true
""")

print("✅ 所有配置文件创建成功！")


✅ 所有配置文件创建成功！


In [10]:
# ========== Cell 9: 在 GAMMAF 目录内创建 TextProcessingManager.py ==========
with open("TextProcessingManager.py", "w") as f:
    f.write("""import torch
from sentence_transformers import SentenceTransformer
from typing import List, Dict, Any

class RoundProcessor:
    def __init__(self, device=None):
        self.device = device if device else ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = SentenceTransformer('all-MiniLM-L12-v2')
        self.model.to(self.device)

    def process_round(self, round_data: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        processed_agents = []

        for agent in round_data:
            processed_agent = agent.copy()

            if "reason" in agent and agent["reason"]:
                st_embedding = self.model.encode(
                    agent["reason"],
                    device=self.device,
                    show_progress_bar=False
                ).tolist()
                processed_agent["st_embedding"] = st_embedding
                processed_agent["tk_embedding"] = [st_embedding] 

            processed_agents.append(processed_agent)

        return processed_agents
""")
print("✅ TextProcessingManager.py 创建成功！")


✅ TextProcessingManager.py 创建成功！


In [11]:
# ========== 在 Cell 10 (运行训练数据生成) 之前插入 ==========
!pip install langchain-openai

# 如果还有其他缺失的依赖，可以直接安装项目自带的 requirements：
# !pip install -r requirements.txt

print("✅ Python 依赖安装完成！")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 46.0 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.23.0
    Uninstalling openai-2.23.0:
      Successfully uninstalled openai-2.23.0
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.15
    Uninstalling langchain-core-1.2.15:
      Successfully uninstalled langchain-core-1.2.15
✅ Python 依赖安装完成！


In [12]:
# ========== Cell 10: 运行训练数据生成 ==========
print("=== 开始运行训练数据生成 ===")
!python TrainDataGeneration.py config-examples/kaggle-generation-config.yaml


=== 开始运行训练数据生成 ===

[TOPOLOGY 1/4] TREE
  seed..............: 42
  planned_questions.: 50
README.md: 7.93kB [00:00, 1.85MB/s]
main/train-00000-of-00001.parquet: 100%|███| 2.31M/2.31M [00:00<00:00, 4.35MB/s]
main/test-00000-of-00001.parquet: 100%|██████| 419k/419k [00:00<00:00, 4.35MB/s]
Q1:   0%|                                                 | 0/3 [00:00<?, ?it/s]

Q2:   0%|                                                 | 0/3 [00:00<?, ?it/s]


Q3:   0%|                                                 | 0/3 [00:00<?, ?it/s]



Q4:   0%|                                                 | 0/3 [00:00<?, ?it/s]




Q5:   0%|                                                 | 0/3 [00:00<?, ?it/s]





Q6:   0%|                                                 | 0/3 [00:00<?, ?it/s]






Q7:   0%|                                                 | 0/3 [00:00<?, ?it/s]







Q8:   0%|                                                 | 0/3 [00:00<?, ?it/s]








Q9:   0%|                   

In [13]:
# ========== Cell 10.5: 为训练数据补充 embeddings ==========
import pickle
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

print("加载 SentenceTransformer 模型...")
st_model = SentenceTransformer('all-MiniLM-L12-v2')

print("为训练数据补充 embeddings...")
with open("/kaggle/working/GAMMAF/data/train-data-kaggle.pkl", "rb") as f:
    train_data = pickle.load(f)

total_agents = 0
for topo_entry in tqdm(train_data, desc="Topologies"):
    for debate in topo_entry["results"]:
        rounds = debate.get("debate_rounds", [])
        for round_data in rounds:
            if not round_data:
                continue
            for agent in round_data:
                reason = agent.get("reason", "")
                if reason and "st_embedding" not in agent:
                    embedding = st_model.encode(reason, show_progress_bar=False).tolist()
                    agent["st_embedding"] = embedding
                    agent["tk_embedding"] = [embedding]
                total_agents += 1

with open("/kaggle/working/GAMMAF/data/train-data-kaggle.pkl", "wb") as f:
    pickle.dump(train_data, f)

print(f"✅ 处理完成，共处理 {total_agents} 个 agent")


加载 SentenceTransformer 模型...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

为训练数据补充 embeddings...


Topologies: 100%|██████████| 4/4 [00:32<00:00,  8.14s/it]

✅ 处理完成，共处理 3024 个 agent


In [14]:
import subprocess, time

!pkill -9 -f "ollama" 2>/dev/null
time.sleep(2)

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(15)

# 验证
!ollama list


]11;?\NAME        ID              SIZE      MODIFIED    
qwen2:7b    dd314f039b9d    4.4 GB    5 hours ago    


In [15]:
# ========== Cell 11.8: 确保 Ollama 在线 ==========
import subprocess, time, requests

# 检查 Ollama 是否还在
try:
    r = requests.post(
        "http://localhost:11434/v1/chat/completions",
        json={"model": "qwen2:7b", "messages": [{"role": "user", "content": "ping"}], "max_tokens": 5},
        timeout=10
    )
    print("✅ Ollama 仍在运行")
except:
    print("⚠️ Ollama 已挂，正在重启...")
    !pkill -9 -f "ollama" 2>/dev/null
    time.sleep(2)
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(15)
    print("✅ Ollama 已重启")
    !ollama list


✅ Ollama 仍在运行


In [16]:
!grep -n "predict" /kaggle/working/GAMMAF/EvaluationDebateLoop.py | head -10


359:        flags, anomaly_scores = defense_model.predict(debate_embeddings, adjacency_matrix, top_k = self.config.top_k_defense)
399:            flags, anomaly_scores = defense_model.predict(debate_embeddings, adjacency_matrix, top_k = self.config.top_k_defense)


In [17]:
# ========== Cell 11: 运行防御基准测试 ==========
import os

# 检查前置条件
assert os.path.exists("/kaggle/working/GAMMAF/data/train-data-kaggle.pkl"), "❌ 训练数据不存在，请先运行 Cell 10"
assert os.path.exists("prompts/prompts_blindguard.json"), "❌ 提示文件不存在"
assert os.path.exists("TextProcessingManager.py"), "❌ TextProcessingManager.py 不存在"

print("=== 开始运行防御基准测试 ===")
!python MainEvaluation.py config-examples/kaggle-evaluation-config.yaml


=== 开始运行防御基准测试 ===
[INFO] Loaded model BlindGuard with embedded configuration from defense_model_train_configs.BlindGuard.
[INFO] Loaded model XG-Guard with embedded configuration from defense_model_train_configs.XG-Guard.

[INFO] Training Model 1/2: BlindGuard
  config........: embedded:defense_model_train_configs.BlindGuard
Loading and processing training data...
Train data processed. Starting training...
Training combined model across all topologies...
Class distribution on combined training data: [608 608]
TRAINING PHASE - Topology: combined
Epoch 1/10 | Train Loss: 4.816364 | Val Loss: 4.614259 [BEST]
Epoch 2/10 | Train Loss: 4.326409 | Val Loss: 4.459824 [BEST]
Epoch 3/10 | Train Loss: 4.127232 | Val Loss: 4.427931 [BEST]
Epoch 4/10 | Train Loss: 4.098259 | Val Loss: 4.417325 [BEST]
Epoch 5/10 | Train Loss: 4.092373 | Val Loss: 4.413480 [BEST]
Epoch 6/10 | Train Loss: 4.087348 | Val Loss: 4.411354 [BEST]
Epoch 7/10 | Train Loss: 4.085141 | Val Loss: 4.409960 [BEST]
Epoch 8/10 | T

In [18]:
# ========== Cell 12: 查看并保存结果 ==========
import json
import os

results_path = "results/evaluation-kaggle.json"
if os.path.exists(results_path):
    with open(results_path, "r") as f:
        results = json.load(f)

    print("=== GAMMAF 评估结果 ===\n")

    for model_name, topo_list in results.items():
        print(f"模型：{model_name}")
        total_q = 0
        total_c = 0
        for topo in topo_list:
            q = topo["total_questions"]
            c = topo["correct_answers"]
            acc = topo["overall_accuracy"]
            total_q += q
            total_c += c
            print(f"  拓扑 {topo['topology']}: {c}/{q} 正确, 准确率 {acc:.4f}")
        if total_q > 0:
            print(f"  总体: {total_c}/{total_q} 正确, 准确率 {total_c/total_q:.4f}")
        print()

    !cp /kaggle/working/GAMMAF/data/train-data-kaggle.pkl /kaggle/working/
    !cp /kaggle/working/GAMMAF/results/evaluation-kaggle.json /kaggle/working/
    print("结果已保存到 Kaggle Output 目录")
else:
    print(f"❌ 结果文件不存在: {results_path}")


=== GAMMAF 评估结果 ===

模型：BlindGuard
  拓扑 tree: 39/50 正确, 准确率 0.7800
  拓扑 chain: 38/50 正确, 准确率 0.7600
  拓扑 star: 38/50 正确, 准确率 0.7600
  拓扑 random: 23/30 正确, 准确率 0.7667
  总体: 138/180 正确, 准确率 0.7667

模型：XG-Guard
  拓扑 tree: 38/50 正确, 准确率 0.7600
  拓扑 chain: 35/50 正确, 准确率 0.7000
  拓扑 star: 40/50 正确, 准确率 0.8000
  拓扑 random: 20/30 正确, 准确率 0.6667
  总体: 133/180 正确, 准确率 0.7389

结果已保存到 Kaggle Output 目录
